# ObsValidator — usage example

This notebook demonstrates the **new JSON Schema 2020-12** API (preferred) and
the **legacy two-arg** API (kept for backward compatibility with tpg).

## New API (preferred)

In [ ]:
from pyaraucaria.obs_plan.obs_plan_parser import ObsPlanParser
from pyaraucaria.ob_validator import ObsValidator

# Default schema (base.yaml) + tpg scheduling kwargs overlay
v = ObsValidator.from_default(overlays=['tpg_overlay'])
print('ready')

In [ ]:
line = ('OBJECT FF_Aql 18:58:14.75 -21:22:14.0 '
        'seq=2/Ic/28,2/V/20 priority=5 cycle=0.08 tag=random_phase '
        'uobi=4d5a6db1 sciprog=t2cep pi=pwielgorski P=1.091787 mV=12.31 '
        'comment="binary and dusty envelope"')

parsed = ObsPlanParser.convert_from_string(line)
ob = ObsValidator.convert_to_obdict(parsed)
result = v.validate_ob(ob, allowed_filters=['V', 'Ic', 'B', 'g', 'r', 'i', 'z'])

print('valid:', result['valid'])
print('data :', result['data'])
for k, ok in result['result'].items():
    if not ok:
        print(f'  ❌ {k}')

In [ ]:
# A deliberately bad line — priority is non-numeric
bad = ('OBJECT VV_Cru 12:23:31.78 -64:28:15.2 seq=2/B/30 '
       'priority=dupa cycle=0.167')
parsed = ObsPlanParser.convert_from_string(bad)
ob = ObsValidator.convert_to_obdict(parsed)
result = v.validate_ob(ob, allowed_filters=['V', 'Ic', 'B'])
print('valid:', result['valid'])
print('result:', result['result'])

## Legacy API (deprecated)

The two-arg constructor `ObsValidator(base_schema, command_rules)` still works and
emits a `DeprecationWarning`. Behaviour is preserved bit-for-bit from the
pre-refactor implementation. This is the path currently used by
`tpg/tpg/telescope_plan_generator.py`. Prefer the new API for new code.

In [ ]:
import warnings

BASE_SCHEMA = ObsValidator.load_schema('base_schema')
COMMAND_RULES = ObsValidator.load_schema('base_rules')
TPG_SCHEMA = ObsValidator.load_schema('tpg_schema')

# The old merge_schemas helper lives in the notebook (kept for reference)
def merge_schemas(base, extra):
    merged = base.copy()
    merged.setdefault('properties', {})
    merged['properties'].update(extra.get('properties', {}))
    return merged

schema = merge_schemas(BASE_SCHEMA, TPG_SCHEMA)

with warnings.catch_warnings():
    warnings.simplefilter('once', DeprecationWarning)
    legacy_v = ObsValidator(schema, COMMAND_RULES)   # ← DeprecationWarning

line = 'OBJECT FF_Aql 18:58:14.75 -21:22:14.0 seq=2/Ic/28'
parsed = ObsPlanParser.convert_from_string(line)
ob = ObsValidator.convert_to_obdict(parsed)
result = legacy_v.validate_ob(ob, allowed_filters=['V', 'Ic'])
print('legacy valid:', result['valid'])